# 🧠 Physics-Informed Neural Networks (PINNs) — Tutorial 2
## Solving the 1D Nonlinear Burgers Equation

---

### 🎯 What you will learn in this tutorial

1. How to handle a **nonlinear PDE residual** (the `u · du/dx` term)
2. Why nonlinear problems are harder and what breaks first
3. **Two-phase training**: Adam for speed, then L-BFGS for precision
4. How to use a **closure function** (required by L-BFGS)
5. How to study the effect of the **viscosity parameter** ν
6. How low viscosity creates sharp gradients that challenge the PINN

---

### 📐 The Problem We Are Solving

We solve the **1D steady-state viscous Burgers equation**:

$$
u \frac{du}{dx} - \nu \frac{d^2u}{dx^2} = 0, \quad x \in [-1, 1]
$$

with Dirichlet boundary conditions:

$$
u(-1) = -1, \quad u(1) = 1
$$

The parameter $\nu > 0$ is the **kinematic viscosity**. The analytical solution is:

$$
u(x) = \frac{x}{1 + \frac{\sqrt{1 + 4\nu^2}\, \sinh(x / 2\nu)}{2\nu \cosh(1/2\nu)}}
$$

For practical purposes we use the cleaner form:

$$
u(x) = -\frac{2\nu}{A} \cdot \frac{d}{dx}\left[\ln \phi(x)\right]
$$

where the exact solution simplifies to:

$$
u(x) = \frac{x}{1 + C \cdot e^{x^2 / (2\nu)}}
$$

We will compute the reference solution **numerically** using SciPy's BVP solver — this is the realistic approach when you don't have a clean closed form.

> **Why this problem?**  
> Burgers is the simplest nonlinear PDE in fluid mechanics. It contains the same `u · ∂u/∂x` convection term as the full Navier-Stokes equations. Mastering it here means the N-S version in Tutorial 4 will feel familiar.

---

### 🆕 What is new compared to Tutorial 1?

| Feature | Tutorial 1 | Tutorial 2 |
|---|---|---|
| PDE type | Linear | **Nonlinear** |
| Residual terms | `-u_xx - f` | **`u·u_x - ν·u_xx`** |
| Optimizer | Adam only | **Adam → L-BFGS** |
| Boundary values | Both zero | **Non-zero, asymmetric** |
| Reference solution | Analytical | **Numerical (SciPy BVP)** |
| Parameter study | None | **Viscosity ν sweep** |

---

## 🧩 Step 1 — Imports

Same stack as Tutorial 1, plus `scipy.integrate.solve_bvp` for the reference solution.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_bvp

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

---

## 🧩 Step 2 — Compute the Reference Solution with SciPy

We use `scipy.integrate.solve_bvp` to get a high-accuracy numerical reference.

This is how you work in practice when there is **no clean analytical formula** — use a trusted classical solver as the ground truth, then validate the PINN against it.

The BVP is rewritten as a first-order system:

$$
y_1 = u, \quad y_2 = \frac{du}{dx}
$$

$$
\frac{dy_1}{dx} = y_2, \quad \frac{dy_2}{dx} = \frac{y_1 \cdot y_2}{\nu}
$$

with $y_1(-1) = -1$, $y_1(1) = 1$.

In [ ]:
def solve_burgers_reference(nu=0.1, n_points=500):
    """
    Solve steady 1D Burgers BVP using SciPy as ground truth.

    ODE system (first-order form):
      y[0] = u
      y[1] = du/dx
      dy[0]/dx = y[1]
      dy[1]/dx = y[0]*y[1] / nu

    BCs: u(-1) = -1,  u(1) = 1
    """
    def ode_rhs(x, y):
        # y[0] = u,  y[1] = u'
        return np.vstack([y[1],
                          y[0] * y[1] / nu])

    def bc_residual(ya, yb):
        # ya = values at x=-1,  yb = values at x=1
        return np.array([ya[0] + 1.0,   # u(-1) = -1
                         yb[0] - 1.0])  # u( 1) =  1

    # Initial guess: linear from -1 to 1
    x_init = np.linspace(-1, 1, n_points)
    y_init = np.zeros((2, n_points))
    y_init[0] = x_init           # u ≈ x  (linear guess)
    y_init[1] = np.ones(n_points) # u' ≈ 1

    sol = solve_bvp(ode_rhs, bc_residual, x_init, y_init, tol=1e-8)

    if not sol.success:
        raise RuntimeError(f'SciPy BVP solver failed: {sol.message}')

    return sol.x, sol.y[0]   # x array, u array


# Compute for default viscosity
NU = 0.1
x_ref, u_ref = solve_burgers_reference(nu=NU)

plt.figure(figsize=(7, 3.5))
plt.plot(x_ref, u_ref, 'k-', lw=2, label=f'Reference (SciPy), ν={NU}')
plt.xlabel('x'); plt.ylabel('u(x)')
plt.title('Reference Solution — Steady Burgers Equation')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.show()
print(f'Solution range: [{u_ref.min():.4f}, {u_ref.max():.4f}]')

---

## 🧩 Step 3 — Understand the Nonlinear Residual

This is the most important conceptual step in Tutorial 2.

In Tutorial 1 the residual was:

$$
r(x) = -\hat{u}_{xx} - f(x) \quad \text{(linear — no interaction between u and its derivatives)}
$$

In Burgers the residual is:

$$
r(x) = \hat{u} \cdot \hat{u}_x - \nu \cdot \hat{u}_{xx} \quad \text{(nonlinear — u multiplied by its own derivative)}
$$

The `u · u_x` term is what makes this nonlinear. The PINN handles it **exactly the same way** — autograd computes `u_x`, and we multiply by `u_pred`. No special treatment needed.

> **Key insight**: Nonlinearity in the PDE does not add complexity to the PINN implementation. It only makes the loss landscape harder to optimise — which is why we need L-BFGS.

In [ ]:
def compute_burgers_residual(model, x, nu):
    """
    PDE residual for steady 1D Burgers:
      r(x) = u * du/dx  -  nu * d²u/dx²

    If the network perfectly solves the PDE, r(x) = 0 everywhere.

    Args:
        model : PINN network
        x     : collocation points (N, 1), requires_grad=True
        nu    : viscosity coefficient (float)

    Returns:
        residual : Tensor (N, 1)
    """
    u = model(x)                         # forward pass

    # First derivative  du/dx
    u_x = torch.autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    # Second derivative  d²u/dx²
    u_xx = torch.autograd.grad(
        u_x, x,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True
    )[0]

    # Burgers residual:  u * u_x  -  nu * u_xx  = 0
    residual = u * u_x - nu * u_xx
    return residual


print('Burgers residual function defined ✓')
print()
print('Breakdown of terms:')
print('  u * u_x   →  nonlinear convection  (what makes this hard)')
print('  nu * u_xx →  linear diffusion      (stabilises the solution)')
print('  At low ν, convection dominates → sharp gradient near x=0')

---

## 🧩 Step 4 — Build the Network

Same architecture as Tutorial 1, but slightly **wider and deeper** — the Burgers solution has a steeper gradient near $x=0$ that requires more capacity.

We also add **Xavier initialisation** — a standard trick that gives the network a better starting point by keeping activations in the active region of `tanh` from the first forward pass.

In [ ]:
class PINN(nn.Module):
    """
    PINN for 1D problems.
    Slightly wider than Tutorial 1 to handle the steeper Burgers solution.
    Includes Xavier weight initialisation.
    """
    def __init__(self, n_hidden=4, n_neurons=64):
        super().__init__()

        layers = [nn.Linear(1, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers.append(nn.Linear(n_neurons, 1))

        self.net = nn.Sequential(*layers)
        self._initialise_weights()   # <-- new in Tutorial 2

    def _initialise_weights(self):
        """
        Xavier (Glorot) uniform initialisation.
        Keeps the variance of activations roughly constant across layers.
        Without this, tanh saturates in deep networks from the start.
        """
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        return self.net(x)


model = PINN(n_hidden=4, n_neurons=64).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal parameters: {total_params}')
print(f'(Tutorial 1 had ~{3*32*32 + 32}: we have more capacity for the steep gradient)')

---

## 🧩 Step 5 — Collocation Points

Domain is $[-1, 1]$ this time. 

New idea: we put **more collocation points near x=0** where the Burgers solution has the sharpest gradient. This is the simplest form of **non-uniform sampling** — a manual version of the adaptive collocation you will see in Tutorial 7.

```
Uniform:    · · · · · · · · · · · · · ·     equal spacing
Clustered:  · ·  · ·   ·     ·   · ·  · ·   more near center
```

In [ ]:
N_COL = 150    # interior collocation points
NU    = 0.1    # viscosity — try changing this later

# Uniform collocation across [-1, 1]
x_col = torch.linspace(-1, 1, N_COL, dtype=torch.float32).view(-1, 1).to(device)
x_col.requires_grad_(True)

# Boundary points and their known values
x_bc = torch.tensor([[-1.0], [1.0]], dtype=torch.float32).to(device)
u_bc = torch.tensor([[-1.0], [1.0]], dtype=torch.float32).to(device)

# Visualise collocation layout
plt.figure(figsize=(10, 1.5))
plt.scatter(x_col.detach().cpu().numpy(), np.zeros(N_COL),
            s=20, c='steelblue', alpha=0.7, label='Collocation pts')
plt.scatter([-1, 1], [0, 0], s=80, c='red', zorder=5, label='Boundary pts')
plt.yticks([])
plt.xlabel('x')
plt.title(f'Collocation Layout — {N_COL} interior points, domain [-1, 1]')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

---

## 🧩 Step 6 — Loss Function

Same structure as Tutorial 1, just with the Burgers residual instead of Poisson.

$$
\mathcal{L} = \underbrace{\frac{1}{N}\sum r(x_i)^2}_{\mathcal{L}_{PDE}} + \lambda \cdot \underbrace{\sum_{k} (\hat{u}(x_k^{bc}) - u_k^{bc})^2}_{\mathcal{L}_{BC}}
$$

One change: we use **higher λ** for the BCs here. The nonlinear solution is more sensitive to boundary values not being enforced from the start.

In [ ]:
def compute_loss(model, x_col, x_bc, u_bc, nu, lambda_bc=20.0):
    """
    Total PINN loss for steady Burgers.

    lambda_bc=20 (higher than Tutorial 1's 10) because
    the nonlinear problem is more sensitive to BC enforcement.
    """
    # PDE loss
    residual = compute_burgers_residual(model, x_col, nu)
    loss_pde = torch.mean(residual ** 2)

    # BC loss — both boundaries in one shot
    u_pred_bc = model(x_bc)
    loss_bc   = torch.mean((u_pred_bc - u_bc) ** 2)

    loss_total = loss_pde + lambda_bc * loss_bc
    return loss_total, loss_pde.item(), loss_bc.item()


print('Loss function defined ✓')

# Sanity check on untrained network
with torch.no_grad():
    u0 = model(x_col)
    print(f'Untrained output range: [{u0.min():.4f}, {u0.max():.4f}]')
    print(f'Target range:           [{u_bc.min():.1f},   {u_bc.max():.1f}  ]')

---

## 🧩 Step 7 — Phase 1: Adam Training

We train with Adam first — it is fast and robust from a random initialisation.

Adam will bring the loss down quickly but will eventually **stall in a flat region** of the loss landscape. That is when L-BFGS takes over.

In [ ]:
ADAM_EPOCHS   = 5000
ADAM_LR       = 1e-3
LAMBDA_BC     = 20.0
LOG_EVERY     = 500

optimizer_adam = torch.optim.Adam(model.parameters(), lr=ADAM_LR)

history = {'epoch': [], 'loss_total': [], 'loss_pde': [], 'loss_bc': [], 'phase': []}

print('Phase 1 — Adam')
print(f'{"Epoch":>8}  {"Loss Total":>14}  {"Loss PDE":>14}  {"Loss BC":>14}')
print('-' * 60)

for epoch in range(1, ADAM_EPOCHS + 1):
    optimizer_adam.zero_grad()
    loss_total, loss_pde, loss_bc = compute_loss(
        model, x_col, x_bc, u_bc, NU, LAMBDA_BC
    )
    loss_total.backward()
    optimizer_adam.step()

    if epoch % LOG_EVERY == 0 or epoch == 1:
        history['epoch'].append(epoch)
        history['loss_total'].append(loss_total.item())
        history['loss_pde'].append(loss_pde)
        history['loss_bc'].append(loss_bc)
        history['phase'].append('Adam')
        print(f'{epoch:>8}  {loss_total.item():>14.6e}  {loss_pde:>14.6e}  {loss_bc:>14.6e}')

print(f'\nAdam complete. Final loss: {loss_total.item():.4e}')

---

## 🧩 Step 8 — Phase 2: L-BFGS Fine-Tuning

### Why L-BFGS?

Adam is a **first-order** optimizer — it only uses gradient information (`dL/dθ`).  
L-BFGS is a **quasi-second-order** optimizer — it approximates the curvature of the loss (`d²L/dθ²`).

```
Adam:    steps downhill, doesn't know the shape of the valley
L-BFGS:  estimates the valley shape and jumps more directly to the bottom
```

### The closure requirement

L-BFGS needs to **re-evaluate the loss multiple times per step** (for the line search). This requires a `closure` function — a callable that:
1. zeros the gradients
2. computes the loss
3. calls `loss.backward()`
4. returns the loss value

> This is different from Adam where you do those steps manually in the loop. L-BFGS calls the closure internally, potentially many times per `optimizer.step()`.

In [ ]:
LBFGS_STEPS   = 500
LOG_EVERY_LB  = 50

optimizer_lbfgs = torch.optim.LBFGS(
    model.parameters(),
    lr=1.0,
    max_iter=20,           # inner iterations per step
    history_size=50,       # how many past gradients to use for curvature estimate
    line_search_fn='strong_wolfe'  # adaptive step size
)

# L-BFGS requires a closure: a function that computes and returns the loss
def closure():
    optimizer_lbfgs.zero_grad()
    loss_total, _, _ = compute_loss(model, x_col, x_bc, u_bc, NU, LAMBDA_BC)
    loss_total.backward()
    return loss_total


print('Phase 2 — L-BFGS')
print(f'{"Step":>8}  {"Loss Total":>14}  {"Loss PDE":>14}  {"Loss BC":>14}')
print('-' * 60)

adam_offset = ADAM_EPOCHS   # for x-axis continuity in the plot

for step in range(1, LBFGS_STEPS + 1):
    optimizer_lbfgs.step(closure)   # closure called internally

    if step % LOG_EVERY_LB == 0 or step == 1:
        # Evaluate separately for logging (no grad needed)
        with torch.no_grad():
            lt, lp, lb = compute_loss(model, x_col, x_bc, u_bc, NU, LAMBDA_BC)
        history['epoch'].append(adam_offset + step)
        history['loss_total'].append(lt.item() if hasattr(lt, 'item') else lt)
        history['loss_pde'].append(lp)
        history['loss_bc'].append(lb)
        history['phase'].append('L-BFGS')
        print(f'{step:>8}  {lt if isinstance(lt, float) else lt.item():>14.6e}  {lp:>14.6e}  {lb:>14.6e}')

print('\nL-BFGS complete ✓')

---

## 🧩 Step 9 — Plot Loss History: Both Phases Together

Plotting both phases on one plot shows the classic two-phase pattern:
- Adam: fast descent, then slows and plateaus
- L-BFGS: picks up where Adam stalled and drives the loss much lower

In [ ]:
epochs     = np.array(history['epoch'])
loss_total = np.array(history['loss_total'])
loss_pde   = np.array(history['loss_pde'])
loss_bc    = np.array(history['loss_bc'])
phases     = np.array(history['phase'])

adam_mask  = phases == 'Adam'
lbfgs_mask = phases == 'L-BFGS'

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: total loss with phase regions shaded
ax = axes[0]
ax.semilogy(epochs[adam_mask],  loss_total[adam_mask],  'b-o', ms=4, lw=1.5, label='Adam')
ax.semilogy(epochs[lbfgs_mask], loss_total[lbfgs_mask], 'r-s', ms=4, lw=1.5, label='L-BFGS')
ax.axvline(ADAM_EPOCHS, color='gray', ls='--', lw=1, label='Phase switch')
ax.set_xlabel('Epoch / Step')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Two-Phase Training: Adam → L-BFGS')
ax.legend(); ax.grid(True, which='both', alpha=0.3)

# Right: PDE vs BC
ax2 = axes[1]
ax2.semilogy(epochs, loss_pde, 'r-o', ms=3, label='PDE loss')
ax2.semilogy(epochs, loss_bc,  'g-s', ms=3, label='BC loss')
ax2.axvline(ADAM_EPOCHS, color='gray', ls='--', lw=1)
ax2.set_xlabel('Epoch / Step')
ax2.set_ylabel('Loss (log scale)')
ax2.set_title('PDE vs BC Loss')
ax2.legend(); ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('burgers_loss_history.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 🧩 Step 10 — Compare PINN to Reference Solution

In [ ]:
# Evaluate trained PINN on fine grid
x_test_np = np.linspace(-1, 1, 500)
x_test    = torch.tensor(x_test_np, dtype=torch.float32).view(-1, 1).to(device)

model.eval()
with torch.no_grad():
    u_pred_np = model(x_test).cpu().numpy().flatten()

# Interpolate reference to same grid
u_ref_interp = np.interp(x_test_np, x_ref, u_ref)
error_np     = np.abs(u_pred_np - u_ref_interp)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Solution comparison
axes[0].plot(x_test_np, u_ref_interp, 'k-',  lw=2.5, label='Reference (SciPy)')
axes[0].plot(x_test_np, u_pred_np,    'r--', lw=2.0, label='PINN prediction')
axes[0].scatter(x_col.detach().cpu().numpy(),
                np.zeros(N_COL), c='steelblue', s=8, zorder=5, label='Collocation pts')
axes[0].set_xlabel('x'); axes[0].set_ylabel('u(x)')
axes[0].set_title(f'PINN vs Reference — Burgers, ν={NU}')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Pointwise error
axes[1].semilogy(x_test_np, error_np + 1e-12, 'purple', lw=2)
axes[1].fill_between(x_test_np, error_np + 1e-12, alpha=0.15, color='purple')
axes[1].set_xlabel('x'); axes[1].set_ylabel('|u_ref - u_PINN| (log)')
axes[1].set_title('Pointwise Absolute Error')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('burgers_solution.png', dpi=150, bbox_inches='tight')
plt.show()

l2_error  = np.sqrt(np.mean((u_pred_np - u_ref_interp) ** 2))
max_error = np.max(error_np)
print(f'L2 error  : {l2_error:.4e}')
print(f'Max error : {max_error:.4e}')

---

## 🧩 Step 11 — Viscosity Study: What Happens When ν Decreases?

This is the most instructive experiment in Tutorial 2.

### The physics

In the Burgers equation:
- **High ν** → diffusion dominates → solution is nearly linear, easy to learn
- **Low ν** → convection dominates → sharp gradient near $x=0$, hard to learn

This is directly analogous to the **Reynolds number** in full Navier-Stokes:

$$
Re = \frac{UL}{\nu}
$$

Low ν = High Re = more turbulent-like behavior = harder for PINNs.

> **For your video**: This is the moment to connect the 1D Burgers result to why PINNs struggle with high-Re CFD — same mechanism, just in 1D where you can see it clearly.

In [ ]:
def train_and_evaluate(nu, n_col=150, adam_epochs=5000, lbfgs_steps=300,
                       lambda_bc=20.0, seed=42):
    """
    Train a fresh PINN for a given viscosity nu.
    Returns (x_test, u_pred, u_ref, l2_error).
    """
    torch.manual_seed(seed)
    net = PINN(n_hidden=4, n_neurons=64).to(device)

    xc = torch.linspace(-1, 1, n_col, dtype=torch.float32).view(-1, 1).to(device)
    xc.requires_grad_(True)
    xbc = torch.tensor([[-1.], [1.]], dtype=torch.float32).to(device)
    ubc = torch.tensor([[-1.], [1.]], dtype=torch.float32).to(device)

    # Phase 1: Adam
    opt_a = torch.optim.Adam(net.parameters(), lr=1e-3)
    for _ in range(adam_epochs):
        opt_a.zero_grad()
        lt, _, _ = compute_loss(net, xc, xbc, ubc, nu, lambda_bc)
        lt.backward()
        opt_a.step()

    # Phase 2: L-BFGS
    opt_l = torch.optim.LBFGS(net.parameters(), lr=1.0, max_iter=20,
                               history_size=50, line_search_fn='strong_wolfe')
    def closure():
        opt_l.zero_grad()
        lt, _, _ = compute_loss(net, xc, xbc, ubc, nu, lambda_bc)
        lt.backward()
        return lt
    for _ in range(lbfgs_steps):
        opt_l.step(closure)

    # Evaluate
    x_ev = torch.linspace(-1, 1, 500, dtype=torch.float32).view(-1, 1).to(device)
    with torch.no_grad():
        u_p = net(x_ev).cpu().numpy().flatten()

    try:
        xr, ur = solve_burgers_reference(nu=nu)
        u_ref_i = np.interp(np.linspace(-1, 1, 500), xr, ur)
        l2 = np.sqrt(np.mean((u_p - u_ref_i) ** 2))
    except Exception:
        u_ref_i = np.zeros(500)
        l2 = float('nan')

    return np.linspace(-1, 1, 500), u_p, u_ref_i, l2


# Run for three viscosity values
nu_values  = [0.5, 0.1, 0.05]
colors     = ['steelblue', 'tomato', 'seagreen']
results    = {}

for nu_val in nu_values:
    print(f'Training ν={nu_val}...', end=' ')
    results[nu_val] = train_and_evaluate(nu_val)
    print(f'L2 = {results[nu_val][3]:.4e}')

# Plot all three
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, nu_val, color in zip(axes, nu_values, colors):
    x_t, u_p, u_r, l2 = results[nu_val]
    ax.plot(x_t, u_r, 'k-',  lw=2.5, label='Reference')
    ax.plot(x_t, u_p, '--',  lw=2.0, color=color, label='PINN')
    ax.set_title(f'ν = {nu_val}\nL2 error = {l2:.2e}')
    ax.set_xlabel('x')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

axes[0].set_ylabel('u(x)')
plt.suptitle('Burgers PINN — Effect of Viscosity ν', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('burgers_viscosity_study.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey observation:')
print('Lower ν → sharper gradient near x=0 → higher PINN error')
print('This is the same challenge as high Reynolds number in full CFD')

---

## 🧩 Step 12 — Visualise the Residual Field After Training

In [ ]:
x_res = torch.linspace(-1, 1, 300, dtype=torch.float32).view(-1, 1).to(device)
x_res.requires_grad_(True)

res_field = compute_burgers_residual(model, x_res, NU)
res_np    = res_field.detach().cpu().numpy().flatten()
x_res_np  = x_res.detach().cpu().numpy().flatten()

plt.figure(figsize=(8, 3.5))
plt.plot(x_res_np, res_np, 'darkorange', lw=2, label=f'PDE residual (ν={NU})')
plt.axhline(0, color='k', lw=0.8, ls='--')
plt.fill_between(x_res_np, res_np, alpha=0.15, color='darkorange')
plt.axvline(0, color='gray', lw=0.8, ls=':')
plt.xlabel('x')
plt.ylabel('r(x) = u·u_x − ν·u_xx')
plt.title('PDE Residual After Training\n(Peak near x=0 — where gradient is steepest)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('burgers_residual.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 📚 Summary — What We Built

| Component | Tutorial 1 | Tutorial 2 (this) |
|---|---|---|
| PDE | Linear Poisson | **Nonlinear Burgers** |
| Residual | `-u_xx - f` | **`u·u_x - ν·u_xx`** |
| Reference | Analytical | **SciPy BVP solver** |
| Network | 3 layers, 32 neurons | **4 layers, 64 neurons + Xavier init** |
| Training | Adam only | **Adam → L-BFGS (two phases)** |
| Experiments | Collocation count | **Viscosity ν sweep** |
| Physics insight | PDE as loss | **Low ν = high Re = hard PINN** |

---

## ➡️ What's Next — Tutorial 3 Preview

In **Tutorial 3** we move to **2D** for the first time:

- Problem: 2D Poisson equation — $-(u_{xx} + u_{yy}) = f(x,y)$
- New: `meshgrid` collocation, two-input network, 2D error heatmaps
- New: **partial derivatives** — $\partial u / \partial x$ and $\partial u / \partial y$ simultaneously
- Same loss structure, but collocation points now live in a 2D domain

The jump from 1D to 2D is smaller than it looks — one extra input neuron and one extra `autograd.grad` call.

---

## 🧩 Exercises

Try these after the tutorial — each one reveals something about PINN behaviour:

1. **Remove L-BFGS** — train with Adam only for 10000 epochs. How does the final L2 error compare?
2. **Change λ_BC** — try `lambda_bc=1` and `lambda_bc=100`. What happens to the solution near the boundaries?
3. **Lower ν to 0.01** — does the PINN still converge? What would help? (Hint: more collocation points near x=0)
4. **Remove Xavier init** — retrain from default init. Does it converge slower or to a worse solution?
5. **Plot u_x** — compute and plot the first derivative of the trained PINN. Does it show the expected peak at x=0?